# 🔢 Reconhecimento de Dígitos Manuscritos com Scikit-Learn

Nesta aula vamos ensinar um computador a **reconhecer números escritos à mão** (de 0 a 9), usando Machine Learning.

### O que vamos aprender:
1. Como os dados de imagem são representados para o computador
2. Como dividir dados em treino e teste
3. Como treinar um modelo de classificação
4. Como avaliar se o modelo está bom
5. Como ver, na prática, onde o modelo erra e por quê

Rode as células em ordem, de cima para baixo.

## 1️⃣ Importando as bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## 2️⃣ Carregando os dados

O scikit-learn já vem com um dataset de exemplo: **1797 imagens de dígitos manuscritos**, cada uma com **8x8 pixels** (64 pixels no total).

Cada imagem tem um **rótulo (label)**: o número que ela representa (0 a 9).

In [ ]:
digits = load_digits()

# Os dados em formato de imagem (para visualizar)
imagens = digits.images   # shape: (1797, 8, 8)

# Os mesmos dados, "achatados" em uma lista de 64 números (para o modelo usar)
X = digits.data           # shape: (1797, 64)

# Os rótulos: qual número cada imagem representa
y = digits.target         # shape: (1797,)

print(f"Temos {X.shape[0]} imagens.")
print(f"Cada imagem tem {X.shape[1]} pixels (8 x 8).")
print(f"Os rótulos possíveis são: {sorted(set(y))}")

### 👀 Vamos VER os dados antes de qualquer coisa

Isso é fundamental: antes de treinar qualquer modelo, sempre olhe os seus dados!

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))

for ax, imagem, label in zip(axes.flat, imagens[:10], y[:10]):
    ax.imshow(imagem, cmap='gray_r')   # gray_r = preto no traço, branco no fundo
    ax.set_title(f"Rótulo: {label}")
    ax.axis('off')

plt.suptitle("Exemplos de dígitos manuscritos no dataset", fontsize=14)
plt.tight_layout()
plt.show()

### 🔍 Como o computador "vê" um dígito

Para nós, é uma imagem. Para o modelo, é só uma lista de **64 números** (a intensidade de cada pixel, de 0 a 16).

Vamos olhar o dígito de número 0 do nosso dataset de duas formas: como imagem e como números.

In [ ]:
indice = 0

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].imshow(imagens[indice], cmap='gray_r')
ax[0].set_title(f"Imagem (rótulo: {y[indice]})")
ax[0].axis('off')

ax[1].imshow(imagens[indice], cmap='gray_r')
for i in range(8):
    for j in range(8):
        valor = int(imagens[indice][i, j])
        cor = 'white' if valor > 8 else 'black'
        ax[1].text(j, i, str(valor), ha='center', va='center', color=cor, fontsize=9)
ax[1].set_title("Os mesmos dados, em números")
ax[1].axis('off')

plt.tight_layout()
plt.show()

print("É exatamente essa grade de 64 números que o modelo recebe como entrada!")

## 3️⃣ Dividindo em treino e teste

Separamos os dados em duas partes:
- **Treino (80%)**: o modelo aprende com esses exemplos
- **Teste (20%)**: usamos para checar se o modelo aprendeu de verdade, com exemplos que ele NUNCA viu

⚠️ Isso é essencial: nunca avalie um modelo com os mesmos dados que ele usou para aprender — seria como dar a prova com as respostas coladas no caderno do aluno.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% para teste
    random_state=42,       # garante que todos tenham o mesmo resultado
    stratify=y              # mantém a proporção de cada dígito equilibrada
)

print(f"Imagens de treino: {len(X_train)}")
print(f"Imagens de teste:  {len(X_test)}")

## 4️⃣ Treinando o modelo

Vamos usar um **SVM (Support Vector Machine)** https://scikit-learn.org/stable/modules/svm.html,
um algoritmo clássico e muito eficaz para esse tipo de problema.

Note como o treino é simples: basta chamar `.fit()` passando os dados e os rótulos corretos.

In [ ]:
modelo = SVC(gamma=0.001)

modelo.fit(X_train, y_train)

print("Modelo treinado! 🎉")

## 5️⃣ Avaliando o modelo

Agora vamos usar o modelo para prever os dígitos do conjunto de **teste** (que ele nunca viu) e comparar com a resposta certa.

In [ ]:
y_pred = modelo.predict(X_test)

acuracia = accuracy_score(y_test, y_pred)
print(f"Acurácia do modelo: {acuracia:.2%}")
print(f"Ou seja: o modelo acertou {int(acuracia * len(y_test))} de {len(y_test)} dígitos!")

### 📊 Relatório de classificação

Mostra a precisão para cada dígito separadamente — alguns números são mais fáceis de confundir que outros (ex: 4 e 9, ou 1 e 7).

In [ ]:
print(classification_report(y_test, y_pred))

### 🧩 Matriz de confusão

Mostra visualmente quais dígitos o modelo confunde entre si. A diagonal principal são os acertos; fora da diagonal são os erros.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues')
plt.title("Matriz de Confusão")
plt.show()

## 6️⃣ O resultado mais palpável: vendo os erros do modelo

Vamos olhar **exatamente quais imagens** o modelo classificou errado, e comparar o que ele *previu* com o que era a resposta *correta*.

Isso ajuda a entender: o modelo errou em algo que até um humano confundiria?

In [ ]:
# Encontrar os índices onde o modelo errou
erros = np.where(y_pred != y_test)[0]
print(f"O modelo errou {len(erros)} de {len(y_test)} imagens.")

# Vamos recuperar as imagens originais (8x8) correspondentes ao conjunto de teste
_, imagens_teste, _, _ = train_test_split(
    imagens, y, test_size=0.2, random_state=42, stratify=y
)

if len(erros) > 0:
    n_mostrar = min(len(erros), 5)
    fig, axes = plt.subplots(1, n_mostrar, figsize=(3 * n_mostrar, 3))
    if n_mostrar == 1:
        axes = [axes]

    for ax, idx in zip(axes, erros[:n_mostrar]):
        ax.imshow(imagens_teste[idx], cmap='gray_r')
        ax.set_title(f"Real: {y_test[idx]} | Previsto: {y_pred[idx]}", color='red')
        ax.axis('off')

    plt.suptitle("Exemplos de erros do modelo", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("O modelo não errou nenhuma imagem neste conjunto de teste! 🎯")

## 7️⃣ Testando com uma imagem escolhida por você

Vamos escolher um índice qualquer do conjunto de teste e ver, lado a lado, a imagem e a previsão do modelo.

👉 **Experimente trocar o valor de `indice_escolhido` para outros números (entre 0 e 359) e rode de novo!**

In [ ]:
indice_escolhido = 7  # 👈 tente mudar este número

imagem_escolhida = X_test[indice_escolhido].reshape(1, -1)  # o modelo espera um "lote" de imagens
previsao = modelo.predict(imagem_escolhida)[0]
real = y_test[indice_escolhido]

plt.figure(figsize=(3, 3))
plt.imshow(X_test[indice_escolhido].reshape(8, 8), cmap='gray_r')
cor_titulo = 'green' if previsao == real else 'red'
plt.title(f"Modelo previu: {previsao}\n(Resposta real: {real})", color=cor_titulo)
plt.axis('off')
plt.show()

## 🎓 Resumo da aula

| Etapa | O que fizemos |
|---|---|
| 1. Dados | Carregamos 1797 imagens de dígitos (8x8 pixels) |
| 2. Divisão | Separamos 80% para treino e 20% para teste |
| 3. Treino | Usamos `.fit()` para o modelo SVM aprender |
| 4. Previsão | Usamos `.predict()` em dados nunca vistos |
| 5. Avaliação | Medimos acurácia, vimos a matriz de confusão e os erros reais |

### 🚀 Desafios
1. Troque o modelo `SVC` por outro, como `KNeighborsClassifier` ou `RandomForestClassifier`. A acurácia muda?
2. Troque o `test_size` para 0.5 (50% para teste). O modelo continua bom?
3. Mude o parâmetro `gamma` do SVM para `0.1` ou `'scale'`. O que acontece?